In [1]:
%pip install jupysql duckdb-engine duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%load_ext sql
%sql duckdb://

Connecting to 'duckdb://'

In [3]:
%%sql
ROLLBACK;

Running query in 'duckdb://'

Success


In [4]:
%%sql
CREATE TABLE clients AS(
    select * from read_csv_auto('../data/clients.csv', encoding='CP1252')
)

Running query in 'duckdb://'

Count


In [5]:
%%sql
CREATE TABLE contrats AS(
    select * from read_csv_auto('../data/contrats-2.csv', encoding='CP1252')
)

Running query in 'duckdb://'

Count


In [6]:
%%sql
CREATE TABLE vehicules AS(
    select * from read_csv_auto('../data/vehicules.csv', encoding='CP1252')
)

Running query in 'duckdb://'

Count


In [7]:
%%sql
# -- Création de la table Référentiel Départements
CREATE TABLE departements (
    cdepso INT,
    dept VARCHAR(50)
);

INSERT INTO departements (cdepso, dept) VALUES
(1, 'Ain'), (2, 'Aisne'), (3, 'Allier'), (4, 'Alpes-de-Haute-Provence'),
(5, 'Hautes-Alpes'), (6, 'Alpes-Maritimes'), (7, 'Ardèche'), (8, 'Ardennes'),
(9, 'Ariège'), (10, 'Aube'), (11, 'Aude'), (12, 'Aveyron'), (13, 'Bouches-du-Rhône'),
(14, 'Calvados'), (15, 'Cantal'), (16, 'Charente'), (17, 'Charente-Maritime'),
(18, 'Cher'), (19, 'Corrèze'), 
(20, 'Corse'), (21, 'Côte-d''Or'), (22, 'Côtes-d''Armor'), (23, 'Creuse'),
(24, 'Dordogne'), (25, 'Doubs'), (26, 'Drôme'), (27, 'Eure'), (28, 'Eure-et-Loir'),
(29, 'Finistère'), (30, 'Gard'), (31, 'Haute-Garonne'), (32, 'Gers'),
(33, 'Gironde'), (34, 'Hérault'), (35, 'Ille-et-Vilaine'), (36, 'Indre'),
(37, 'Indre-et-Loire'), (38, 'Isère'), (39, 'Jura'), (40, 'Landes'),
(41, 'Loir-et-Cher'), (42, 'Loire'), (43, 'Haute-Loire'), (44, 'Loire-Atlantique'),
(45, 'Loiret'), (46, 'Lot'), (47, 'Lot-et-Garonne'), (48, 'Lozère'),
(49, 'Maine-et-Loire'), (50, 'Manche'), (51, 'Marne'), (52, 'Haute-Marne'),
(53, 'Mayenne'), (54, 'Meurthe-et-Moselle'), (55, 'Meuse'), (56, 'Morbihan'),
(57, 'Moselle'), (58, 'Nièvre'), (59, 'Nord'), (60, 'Oise'), (61, 'Orne'),
(62, 'Pas-de-Calais'), (63, 'Puy-de-Dôme'), (64, 'Pyrénées-Atlantiques'),
(65, 'Hautes-Pyrénées'), (66, 'Pyrénées-Orientales'), (67, 'Bas-Rhin'),
(68, 'Haut-Rhin'), (69, 'Rhône'), (70, 'Haute-Saône'), (71, 'Saône-et-Loire'),
(72, 'Sarthe'), (73, 'Savoie'), (74, 'Haute-Savoie'), (75, 'Paris'),
(76, 'Seine-Maritime'), (77, 'Seine-et-Marne'), (78, 'Yvelines'),
(79, 'Deux-Sèvres'), (80, 'Somme'), (81, 'Tarn'), (82, 'Tarn-et-Garonne'),
(83, 'Var'), (84, 'Vaucluse'), (85, 'Vendée'), (86, 'Vienne'),
(87, 'Haute-Vienne'), (88, 'Vosges'), (89, 'Yonne'), (91, 'Essonne'),
(92, 'Hauts-de-Seine'), (93, 'Seine-Saint-Denis'), (94, 'Val-de-Marne'),
(95, 'Val-d''Oise'), (97, 'Outre-mer'), (98, 'Outre-mer');

Running query in 'duckdb://'

Count


In [8]:
%%sql
CREATE TABLE base_clients AS
WITH clients_transform AS (
    SELECT 
        *,
        -- Reconstitution de la date de naissance (YYYY-MM-DD)
        MAKE_DATE(anaiso, mnaiso, jnaiso) AS date_nais,
        (2025 - anaiso) AS age,
        (EXTRACT(YEAR FROM CURRENT_DATE) - anaiso) AS age2,
        (CURRENT_DATE - MAKE_DATE(anaiso, mnaiso, jnaiso)) AS age3,
        (EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(anaiso, mnaiso, jnaiso)))) AS age4
    FROM clients
)
SELECT 
    *,
    -- Tranches d'âge
    CASE 
        WHEN age >= 18 AND age < 30 THEN '- de 30'
        WHEN age >= 30 AND age <= 50 THEN '30-50'
        WHEN age > 50 AND age < 65 THEN '51 - 64'
        WHEN age >= 65 THEN '65 et +'
        ELSE 'erreur'
    END AS tr_age,

    -- Recodage Classe d'âge (<30 / >30)
    CASE 
        WHEN age <= 30 THEN 'moins 30 ans'
        ELSE 'plus 30 ans'
    END AS classe_age,

    -- Libellé Sexe
    CASE 
        WHEN sexsoc = 1 THEN 'homme'
        WHEN sexsoc = 2 THEN 'femme'
        WHEN sexsoc = 3 THEN 'sociétaire'
        ELSE 'autre'
    END AS sexe,

    -- Libellé Profession (CSP)
    CASE 
        WHEN cspsoc = 11 THEN 'agriculteur'
        WHEN cspsoc = 12 THEN 'artisan'
        WHEN cspsoc = 13 THEN 'entrepreneur'
        WHEN cspsoc = 14 THEN 'libérale'
        WHEN cspsoc = 20 THEN 'cadre'
        WHEN cspsoc = 30 THEN 'intermédiaire'
        WHEN cspsoc = 62 THEN 'étudiant'
        WHEN cspsoc IN (40, 50) THEN 'employé'
        WHEN cspsoc = 61 THEN 'retraité'
        WHEN cspsoc = 70 THEN 'personne morale'
        WHEN cspsoc = 80 THEN 'non connu'
        WHEN cspsoc IN (63, 64, 65) THEN 'sans activité'
        ELSE 'autre'
    END AS profession
FROM clients_transform;

Running query in 'duckdb://'

Count


In [9]:
%%sql
CREATE TABLE contrats_enrichis AS
SELECT 
    *,
    -- Libellé Usage
    CASE 
        WHEN usagco1 = 0 THEN 'domicile-travail'
        WHEN usagco1 = 1 THEN 'promenade'
        WHEN usagco1 = 3 THEN 'professionnel'
        ELSE 'autre'
    END AS usage,

    -- Création de la variable Formule
    CASE 
        WHEN g01co = 1 AND g03co = 1 AND g09co = 1 THEN 'DTA'
        WHEN g01co = 1 AND g03co = 1 THEN 'RCvol'
        WHEN g01co = 1 THEN 'RC'
        WHEN g22co = 1 THEN 'RCHC'
        ELSE NULL
    END AS formule
FROM contrats
WHERE (g01co = 1 OR g22co = 1); -- Équivalent du "else delete" dans SAS

Running query in 'duckdb://'

Count


In [10]:
%%sql
CREATE TABLE base_finale AS
SELECT DISTINCT
    c.nucon,
    c.nusoc,
    c.usage,
    c.formule,
    c.acreco,
    TRY_CAST(REPLACE(c.prmaco, ',', '.') AS DOUBLE) AS prmaco, 
    v.pfco,
    c.usagco1,
    c.cateco,
    v.utimot,
    cl.age,
    cl.tr_age,
    cl.classe_age,
    cl.sexe,
    cl.sitmat,
    cl.profession,
    d.dept,
    (2025 - c.acreco) AS anciennete_contrat,
    (TRY_CAST(REPLACE(c.prmaco, ',', '.') AS DOUBLE) / NULLIF(v.pfco, 0)) AS prime_par_puissance
FROM contrats_enrichis c
INNER JOIN vehicules v
    ON c.nucon = v.nucon AND c.nusoc = v.nusoc
INNER JOIN base_clients cl
    ON c.nusoc = cl.nusoc
LEFT JOIN departements d
    ON cl.cdepso = d.cdepso;

Running query in 'duckdb://'

Count


In [11]:
%%sql
SELECT 
    COUNT(age) AS nb_observations,
    AVG(age) AS age_moyen,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY age) AS age_median
FROM base_clients;

Running query in 'duckdb://'

nb_observations,age_moyen,age_median
1257,44.53301511535402,39.0


In [12]:
%%sql
SELECT 
    sexe,
    tr_age,
    COUNT(*) AS count
FROM base_clients
GROUP BY sexe, tr_age
ORDER BY sexe, tr_age;

Running query in 'duckdb://'

sexe,tr_age,count
femme,- de 30,9
femme,30-50,76
femme,51 - 64,27
femme,65 et +,8
homme,- de 30,79
homme,30-50,687
homme,51 - 64,230
homme,65 et +,97
sociétaire,30-50,1
sociétaire,51 - 64,4


In [13]:
%%sql
SELECT 
    acreco,
    COUNT(CASE WHEN tr_age = '- de 30' THEN 1 END) AS "- de 30",
    COUNT(CASE WHEN tr_age = '30-50' THEN 1 END) AS "30-50",
    COUNT(CASE WHEN tr_age = '51 - 64' THEN 1 END) AS "51 - 64",
    COUNT(CASE WHEN tr_age = '65 et +' THEN 1 END) AS "65 et +"
FROM base_finale
GROUP BY acreco
ORDER BY acreco;

Running query in 'duckdb://'

acreco,- de 30,30-50,51 - 64,65 et +
2023,22,362,120,117
2024,92,568,250,208


In [14]:
%%sql
SELECT 
    COUNT(formule) AS n_formule,
    COUNT(*) - COUNT(formule) AS nmiss_formule,
    COUNT(prmaco) AS n_prmaco,
    COUNT(*) - COUNT(prmaco) AS nmiss_prmaco
FROM base_finale;

Running query in 'duckdb://'

n_formule,nmiss_formule,n_prmaco,nmiss_prmaco
1739,0,1739,0


In [15]:
%%sql 
CREATE TABLE base_data_viz_pb AS
SELECT DISTINCT
    c.nucon,
    c.nusoc,
    # Ajout de la vraie date pour l'intelligence temporelle dans Power BI
    MAKE_DATE(c.acreco, c.mcreco, c.jcreco) AS date_souscription,
    c.mcreco AS mois_num,
    MONTHNAME(MAKE_DATE(c.acreco, c.mcreco, c.jcreco)) AS mois_nom,
    CONCAT('T', QUARTER(MAKE_DATE(c.acreco, c.mcreco, c.jcreco))) AS trimestre,
    CASE 
        WHEN c.mcreco IN (6, 7, 8) THEN 'Été'
        WHEN c.mcreco IN (9, 10, 11) THEN 'Automne'
        WHEN c.mcreco IN (12, 1, 2) THEN 'Hiver'
        ELSE 'Printemps'
    END AS saison,
    c.usage,
    c.formule,
    c.acreco,
    TRY_CAST(REPLACE(c.prmaco, ',', '.') AS DOUBLE) AS prmaco, 
    v.pfco,
    c.usagco1,
    c.cateco,
    v.utimot,
    cl.age,
    cl.tr_age,
    cl.classe_age,
    cl.sexe,
    cl.sitmat,
    cl.profession,
    d.dept,
    (2025 - c.acreco) AS anciennete_contrat,
    (TRY_CAST(REPLACE(c.prmaco, ',', '.') AS DOUBLE) / NULLIF(v.pfco, 0)) AS prime_par_puissance
FROM contrats_enrichis c
INNER JOIN vehicules v
    ON c.nucon = v.nucon AND c.nusoc = v.nusoc
INNER JOIN base_clients cl
    ON c.nusoc = cl.nusoc
LEFT JOIN departements d
    ON cl.cdepso = d.cdepso;

Running query in 'duckdb://'

Count


In [16]:
%%sql
COPY base_data_viz_pb TO '../data/base_data_viz_pb.parquet' (FORMAT PARQUET);

Running query in 'duckdb://'

Count
